<img src="../../../aulas/notebooks/imgs/unifor_logo.png" width="400">
<br>
<b>
<font size="6" face="arial" color="blue">
    Graduação em Ciência da Computação
</font>
</b>
<br
<b>
<font size="4" face="arial">
    Disciplina: Métodos Quantitativos para Computação
</font>
</b>

**Orientador: Prof. Me. Ricardo Carubbi** <br>
*Docente da Graduação e Pós-Graduação em Ciência de Dados e Inteligência Artificial*<br>
*Laboratório de Ciência de Dados e Inteligência Artificial*<br>
*Universidade de Fortaleza*<br>

<p dir="ltr" style="text-align: left;">
    <strong>Links:</strong>
    <a href="https://www2.unifor.br/controle_pesquisa/pesquisarprofessor.do?actionParameter=prepareUpdate&amp;p_tp_ambiente=2&amp;p_tp_chamada=1&amp;p_tp_apresentacao=1&amp;cdPesquisador=767686193" target="_blank">Unifor</a|
    <a href="http://lattes.cnpq.br/5738786447903616" target="_blank">Lattes</a|
    <a href="https://unifor.br/web/pesquisa-inovacao/ncdia" target="_blank">NCDIA</a|
    <a href="https://github.com/carubbi/" target="_blank">Github</a>
</p>

# **Trabalho da Unidade 3**: Regressão Linear e Salários na NBA

Este é o notebook modelo do trabalho avaliativo sobre **regressão linear simples** e introdução à **regressão linear múltipla**, preparado para uso no Google Colab.

O trabalho dá continuidade ao T1: em vez de descrever salários e desempenho em duas temporadas, o grupo agora usará **todas as temporadas disponíveis do mesmo time** para investigar a associação entre desempenho e salário.

Regras principais:

- use o time definido para seu grupo;
- use todas as temporadas disponíveis desse time no dataset;
- a unidade de análise é `jogador-temporada`;
- o modelo principal obrigatório é `salario_usd ~ pontos`;
- o modelo complementar deve ser uma regressão múltipla escolhida e justificada pelo grupo;
- nos relatórios dos modelos, interprete coeficiente, erro padrão, estatística t, valor-p `p(t)` e intervalo de confiança de 95% para $\alpha = 0{,}05$;
- não interprete os resultados como causalidade;
- na análise estatística, use apenas `pandas`, `numpy`, `matplotlib` e `statsmodels`; a preparação usa `gdown` apenas para baixar o dataset no Colab;
- não use `seaborn`;
- escreva fórmulas somente em blocos com `$$`.


## **Identificação do Grupo**

Use o mesmo time associado ao grupo no T1.

| Grupo | Time | Sigla |
| --- | --- | --- |
| Grupo C | New York Knicks | `NY` |
| Grupo B | Milwaukee Bucks | `MIL` |
| Grupo A | Minnesota Timberwolves | `MIN` |
| Grupo D | Golden State Warriors | `GS` |

### **Resposta do Grupo**

Preencha os nomes dos integrantes, o grupo e a sigla do time.

In [ ]:
# Preencha com as informações do seu grupo.
# Use exatamente uma das opções: "Grupo A", "Grupo B", "Grupo C" ou "Grupo D".
# Use exatamente a sigla correspondente ao grupo.

GRUPO = None
TIME = None

GRUPO, TIME


In [ ]:
grupos_times = {
    "Grupo A": "MIN",
    "Grupo B": "MIL",
    "Grupo C": "NY",
    "Grupo D": "GS",
}

assert GRUPO in grupos_times, "GRUPO deve ser um dos grupos definidos na tabela."
assert TIME == grupos_times[GRUPO], "TIME não corresponde ao grupo informado."
print(f"Grupo validado: {GRUPO} - {TIME}")


## **Preparação dos Dados**

Objetivo: carregar o dataset completo e filtrar apenas o time do grupo, usando todas as temporadas disponíveis.

A unidade de análise é `jogador-temporada`: cada linha representa um jogador em uma temporada por um time.

### **Imports e Configurações**

Execute esta célula antes das demais para carregar as bibliotecas e definir a configuração padrão dos gráficos.

In [ ]:
# Configuração do ambiente analítico.
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from gdown import download

plt.style.use("default")
plt.rcParams["figure.figsize"] = (8, 5)
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25
plt.rcParams["grid.linestyle"] = "--"
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

### **Funções Auxiliares**

Execute esta célula para disponibilizar a função de download do arquivo do Google Drive.

In [ ]:
# Definir a função download_from_drive.
def download_from_drive(link, filename):

    # Criar a pasta para download.
    foldername = filename.split("/")[0]
    if not os.path.exists(foldername):
        os.makedirs(foldername)
    else:
        print(f"A pasta {foldername} já existe!")

    # Extrair o ID do arquivo a partir do link do Google Drive.
    file_id = link.split("/")[5]
    # Criar a URL de download usando o ID do arquivo.
    url = f"https://drive.google.com/uc?id={file_id}"
    download(url, filename, quiet=False)

### **Carregamento do Dataset**

Execute esta célula para baixar o dataset, salvar o arquivo na pasta `dataset/` e carregar os dados em `df`.

In [ ]:
url_dataset = "https://drive.google.com/file/d/1WOzabGvTmyTcjWIsDRO0wjrybwLubnyu/view?usp=share_link"
dst_dataset = "dataset/nba_stats_preprocessed.csv"

download_from_drive(url_dataset, dst_dataset)
df = pd.read_csv(dst_dataset)
df.head()

In [ ]:
# Exibe informações sobre o dataset.
df.info()

In [ ]:
# Crie o dataframe dados com apenas as linhas do time definido em TIME.
# Ao final, dados deve conter todas as temporadas disponíveis desse time.

dados = None

dados.shape

In [ ]:
assert isinstance(dados, pd.DataFrame), "dados deve ser um DataFrame."
assert len(dados) > 0, "dados está vazio. Verifique o filtro do time."
assert dados["sigla_time"].nunique() == 1, "dados deve conter apenas um time."
assert dados["sigla_time"].iloc[0] == TIME, "dados contém time diferente de TIME."
assert dados["temporada"].nunique() >= 10, "use todas as temporadas disponíveis do time; há poucas temporadas no filtro."
print("Filtro validado.")
print(dados[["temporada", "sigla_time", "nome_jogador", "salario_usd", "pontos"]].head())

In [ ]:
# Apresente verificações básicas da base filtrada:
# - número de linhas;
# - primeira e última temporada;
# - quantidade de temporadas;
# - valores ausentes nas variáveis centrais.

resumo_base = None
ausentes_centrais = None

resumo_base, ausentes_centrais

In [ ]:
assert resumo_base is not None, "preencha resumo_base."
assert ausentes_centrais is not None, "preencha ausentes_centrais."
print("Verificações iniciais registradas.")

## **Pergunta de Pesquisa**

No T1, o grupo descreveu salários e indicadores de desempenho do time em duas temporadas. Nesta unidade, a pergunta passa a ser modelada por regressão:

**Jogadores com maior pontuação total na temporada tendem a apresentar salários maiores no time analisado?**

O modelo principal será:

$$
salario\_usd_i = \beta_0 + \beta_1 pontos_i + \varepsilon_i
$$

A análise é observacional. Portanto, os resultados indicam **associação estatística**, não causalidade.

## **Descrição Inicial de Salário e Pontos**

Antes do ajuste da regressão, descreva a variável resposta (`salario_usd`) e a variável preditora principal (`pontos`).

Gráficos obrigatórios nesta etapa:

- histograma de `salario_usd`;
- histograma de `pontos`;
- diagrama de dispersão `pontos` × `salario_usd`.

In [ ]:
# Crie a base do modelo principal com salario_usd e pontos sem valores ausentes.
# Registre quantas linhas foram removidas pelo dropna.

variaveis_modelo_1 = ["salario_usd", "pontos"]
n_antes_modelo_1 = None
modelo_1_df = None
n_depois_modelo_1 = None
n_removidas_modelo_1 = None

n_antes_modelo_1, n_removidas_modelo_1, n_depois_modelo_1

In [ ]:
assert isinstance(modelo_1_df, pd.DataFrame), "modelo_1_df deve ser um DataFrame."
assert set(variaveis_modelo_1).issubset(modelo_1_df.columns), "modelo_1_df deve conter salario_usd e pontos."
assert n_antes_modelo_1 is not None, "preencha n_antes_modelo_1."
assert n_depois_modelo_1 is not None, "preencha n_depois_modelo_1."
assert n_removidas_modelo_1 == n_antes_modelo_1 - n_depois_modelo_1, "confira a contabilidade do dropna."
assert len(modelo_1_df) == n_depois_modelo_1, "n_depois_modelo_1 deve ser igual ao tamanho de modelo_1_df."
assert len(modelo_1_df) >= 30, "a amostra do modelo principal deve ter pelo menos 30 casos."
assert not modelo_1_df[variaveis_modelo_1].isna().any().any(), "modelo_1_df não deve conter ausentes nas variáveis do modelo."
print("Base do modelo principal validada.")

In [ ]:
# Calcule estatísticas descritivas de salario_usd e pontos.

descritivas_modelo_1 = None

descritivas_modelo_1

In [ ]:
assert descritivas_modelo_1 is not None, "preencha descritivas_modelo_1."
print("Estatísticas descritivas registradas.")

In [ ]:
# Faça um histograma de salario_usd.
# Identifique eixo x, eixo y e título.

# SEU CÓDIGO AQUI

### **Interpretação da Distribuição dos Salários**

Responda em 4 a 6 linhas:

- a distribuição dos salários parece simétrica ou assimétrica?
- há salários muito superiores à maioria dos casos?
- por que isso pode afetar uma regressão linear?

In [ ]:
# Faça um histograma de pontos.
# Identifique eixo x, eixo y e título.

# SEU CÓDIGO AQUI

### **Interpretação da Distribuição dos Pontos**

Responda em 4 a 6 linhas:

- a distribuição de `pontos` parece simétrica ou assimétrica?
- há muitos jogadores-temporada com baixa pontuação?
- há jogadores-temporada com pontuação muito alta?
- a variável `pontos` apresenta variação suficiente para ser usada como preditora no modelo?

In [ ]:
# Faça um diagrama de dispersão entre pontos e salario_usd.
# Use pontos no eixo x e salario_usd no eixo y.
# Identifique eixo x, eixo y e título.

# SEU CÓDIGO AQUI

### **Interpretação Visual da Associação**

Responda em 4 a 6 linhas:

- a associação visual parece positiva, negativa ou ausente?
- há grande dispersão dos salários para jogadores com pontuação semelhante?
- há observações extremas que podem influenciar a reta?

## **Modelo 1: Regressão Linear Simples**

Ajuste o modelo principal obrigatório:

$$
salario\_usd_i = \beta_0 + \beta_1 pontos_i + \varepsilon_i
$$

Interprete o coeficiente angular como associação média entre pontuação total e salário, na escala original de dólares.

Além do coeficiente, o relatório deve ser interpretado com $\alpha = 0{,}05$:

- **erro padrão**: mede a imprecisão da estimativa do coeficiente;
- **estatística t**: razão entre coeficiente estimado e erro padrão;
- **valor-p `p(t)`**: evidência contra a hipótese nula de coeficiente igual a zero;
- **IC 95%**: faixa de valores compatíveis com os dados para o coeficiente, sob os pressupostos do modelo.

Use essas métricas como evidência estatística, não como prova de causalidade.


In [ ]:
# Ajuste o modelo salario_usd ~ pontos usando statsmodels.
# Use sm.add_constant para incluir o intercepto.

X_pontos = None
fit_simples = None

print(fit_simples.summary())

In [ ]:
assert X_pontos is not None, "crie X_pontos."
assert fit_simples is not None, "crie fit_simples."
assert hasattr(fit_simples, "params"), "fit_simples deve ser um resultado de modelo statsmodels."
assert "pontos" in fit_simples.params.index, "o modelo simples deve conter o coeficiente de pontos."
print("Modelo simples validado.")

In [ ]:
# Organize os principais resultados do modelo simples em uma tabela.
# A tabela deve conter uma linha por termo do modelo: const e pontos.
# A tabela deve conter: termo, coeficiente, erro_padrao, estatistica_t, p_t, ic_95_inf e ic_95_sup.
# Use alpha=0.05 no intervalo de confiança.
# Dicas: fit_simples.params, fit_simples.bse, fit_simples.tvalues,
# fit_simples.pvalues e fit_simples.conf_int(alpha=0.05).

resultados_simples = None

resultados_simples


In [ ]:
assert isinstance(resultados_simples, pd.DataFrame), "resultados_simples deve ser um DataFrame."
colunas_resultados_modelo = [
    "termo",
    "coeficiente",
    "erro_padrao",
    "estatistica_t",
    "p_t",
    "ic_95_inf",
    "ic_95_sup",
]
for coluna in colunas_resultados_modelo:
    assert coluna in resultados_simples.columns, f"resultados_simples deve conter {coluna}."
assert {"const", "pontos"}.issubset(set(resultados_simples["termo"])), "inclua const e pontos em resultados_simples."
for coluna in colunas_resultados_modelo[1:]:
    assert pd.api.types.is_numeric_dtype(resultados_simples[coluna]), f"{coluna} deve ser numérica."
assert resultados_simples["p_t"].between(0, 1).all(), "p_t deve estar entre 0 e 1."
print("Tabela do modelo simples validada.")


### **Interpretação do Modelo Simples**

Responda em 8 a 12 linhas:

- qual é o sinal do coeficiente de `pontos`?
- como interpretar esse coeficiente na escala de dólares?
- o erro padrão do coeficiente de `pontos` sugere estimativa precisa ou imprecisa? Justifique pela escala do coeficiente;
- qual é a estatística t de `pontos` e o que ela compara?
- com $\alpha = 0{,}05$, o valor-p `p(t)` de `pontos` indica evidência estatística de associação linear diferente de zero?
- o IC 95% de `pontos` inclui zero? A conclusão é coerente com o valor-p?
- o valor de $R^2$ indica forte ou fraca capacidade explicativa?
- a interpretação permite afirmar causalidade? Por quê?


In [ ]:
# Gere valores ajustados ao longo da escala observada de pontos.
# A tabela deve conter mean, mean_se, mean_ci_lower e mean_ci_upper.
# Atenção: este IC 95% é da média ajustada, não dos coeficientes do modelo.

pontos_grid = None
pontos_grid_X = None
pred_simples = None

tabela_pred_simples = None

tabela_pred_simples.head().round(2)


In [ ]:
assert isinstance(tabela_pred_simples, pd.DataFrame), "tabela_pred_simples deve ser um DataFrame."
for coluna in ["pontos", "mean", "mean_se", "mean_ci_lower", "mean_ci_upper"]:
    assert coluna in tabela_pred_simples.columns, f"tabela_pred_simples deve conter {coluna}."
print("Tabela de valores ajustados validada.")

### **Interpretação da Tabela de Valores Ajustados**

Defina e interprete:

- `mean`;
- `mean_se`;
- `mean_ci_lower`;
- `mean_ci_upper`.

Explique o que acontece com o salário médio ajustado quando `pontos` aumenta.

Não confunda este intervalo de confiança da **média ajustada** com o IC 95% dos **coeficientes** apresentado em `resultados_simples`.


In [ ]:
# Faça o gráfico da reta ajustada com IC 95% da média.
# Use Matplotlib.
# Inclua título, rótulos dos eixos e legenda.

# SEU CÓDIGO AQUI

### **Interpretação da Reta Ajustada**

Responda em 5 a 7 linhas:

- a reta confirma a tendência visual observada no diagrama de dispersão?
- o IC 95% da média ajustada é mais estreito em quais regiões?
- a largura do IC sugere maior incerteza para quais valores de `pontos`?
- a evidência do coeficiente de `pontos` em `resultados_simples` é coerente com a inclinação visual da reta?
- quais cuidados devem ser tomados ao interpretar salários muito altos?


## **Triagem de Variáveis Para o Modelo Múltiplo**

O modelo complementar será escolhido pelo grupo. A escolha das variáveis faz parte da avaliação.

O grupo pode testar diferentes variáveis, mas deve explicar os riscos da escolha. Alguns alertas:

- `ranking_salarial` tem risco de circularidade, pois deriva da ordenação dos salários;
- variáveis como arremessos convertidos ou tentados podem ser redundantes com `pontos`;
- variáveis com muitos ausentes podem reduzir a amostra;
- variáveis muito correlacionadas entre si podem dificultar a interpretação dos coeficientes parciais;
- `temporada` pode capturar mudanças históricas de salários e teto salarial, não apenas desempenho.

In [ ]:
# Defina uma lista de variáveis candidatas para avaliar.
# Inclua pelo menos 5 variáveis candidatas além de salario_usd.

variaveis_candidatas = None

variaveis_candidatas

In [ ]:
assert isinstance(variaveis_candidatas, list), "variaveis_candidatas deve ser uma lista."
assert len(variaveis_candidatas) >= 5, "inclua pelo menos 5 variáveis candidatas."
assert "salario_usd" not in variaveis_candidatas, "salario_usd é a resposta, não deve estar entre candidatas explicativas."
for coluna in variaveis_candidatas:
    assert coluna in dados.columns, f"{coluna} não existe em dados."
    assert pd.api.types.is_numeric_dtype(dados[coluna]), f"{coluna} deve ser numérica para esta triagem."
print("Lista de variáveis candidatas validada.")

In [ ]:
# Monte uma tabela de triagem das variáveis candidatas.
# A tabela deve conter: variavel, ausentes, corr_salario, corr_pontos e risco_interpretacao.
# Quando a variável candidata for pontos, registre corr_pontos = 1.0.

triagem_variaveis = None

triagem_variaveis

In [ ]:
assert isinstance(triagem_variaveis, pd.DataFrame), "triagem_variaveis deve ser um DataFrame."
for coluna in ["variavel", "ausentes", "corr_salario", "corr_pontos", "risco_interpretacao"]:
    assert coluna in triagem_variaveis.columns, f"triagem_variaveis deve conter {coluna}."
assert len(triagem_variaveis) >= 5, "a tabela de triagem deve avaliar pelo menos 5 variáveis."
print("Tabela de triagem validada.")

### **Justificativa da Escolha das Variáveis**

Responda em 8 a 12 linhas:

- quais variáveis foram candidatas?
- quais variáveis foram descartadas e por quê?
- há risco de circularidade, redundância ou muitos ausentes?
- por que as variáveis escolhidas ajudam a interpretar salários?
- a escolha melhora a interpretação ou apenas aumenta o $R^2$?

## **Modelo 2: Regressão Linear Múltipla Escolhida Pelo Grupo**

O modelo múltiplo deve conter `pontos` e pelo menos mais uma variável explicativa escolhida pelo grupo.

Exemplo geral:

$$
salario\_usd_i = \beta_0 + \beta_1 pontos_i + \beta_2 X_i + \varepsilon_i
$$

ou:

$$
salario\_usd_i = \beta_0 + \beta_1 pontos_i + \beta_2 X_i + \beta_3 Z_i + \varepsilon_i
$$

No modelo múltiplo, os coeficientes são **associações parciais**: cada coeficiente mede a associação entre sua variável e `salario_usd` mantendo constantes as demais variáveis incluídas no modelo.

Para cada coeficiente, interprete erro padrão, estatística t, valor-p `p(t)` e IC 95% com $\alpha = 0{,}05$. Uma variável pode aumentar o $R^2$ e ainda assim ter interpretação fraca, redundante ou instável.


In [ ]:
# Defina as variáveis explicativas do modelo múltiplo.
# A lista deve conter pontos e pelo menos mais uma variável.

variaveis_modelo_multiplo = None

variaveis_modelo_multiplo

In [ ]:
assert isinstance(variaveis_modelo_multiplo, list), "variaveis_modelo_multiplo deve ser uma lista."
assert "pontos" in variaveis_modelo_multiplo, "o modelo múltiplo deve conter pontos."
assert len(variaveis_modelo_multiplo) >= 2, "inclua pelo menos uma variável além de pontos."
assert "salario_usd" not in variaveis_modelo_multiplo, "salario_usd é a resposta, não deve ser preditora."
for coluna in variaveis_modelo_multiplo:
    assert coluna in dados.columns, f"{coluna} não existe em dados."
    assert pd.api.types.is_numeric_dtype(dados[coluna]), f"{coluna} deve ser numérica."
if "ranking_salarial" in variaveis_modelo_multiplo:
    print("ATENÇÃO: ranking_salarial tem risco de circularidade por derivar da ordenação dos salários.")
print("Variáveis do modelo múltiplo registradas.")

In [ ]:
# Crie a base do modelo múltiplo sem valores ausentes nas variáveis usadas.
# Registre quantas linhas foram removidas pelo dropna.

variaveis_modelo_2 = ["salario_usd"] + variaveis_modelo_multiplo
n_antes_modelo_2 = None
modelo_2_df = None
n_depois_modelo_2 = None
n_removidas_modelo_2 = None

n_antes_modelo_2, n_removidas_modelo_2, n_depois_modelo_2

In [ ]:
assert isinstance(modelo_2_df, pd.DataFrame), "modelo_2_df deve ser um DataFrame."
assert set(variaveis_modelo_2).issubset(modelo_2_df.columns), "modelo_2_df deve conter todas as variáveis do modelo."
assert n_removidas_modelo_2 == n_antes_modelo_2 - n_depois_modelo_2, "confira a contabilidade do dropna."
assert len(modelo_2_df) == n_depois_modelo_2, "n_depois_modelo_2 deve ser igual ao tamanho de modelo_2_df."
assert len(modelo_2_df) >= 30, "a amostra do modelo múltiplo deve ter pelo menos 30 casos."
assert not modelo_2_df[variaveis_modelo_2].isna().any().any(), "modelo_2_df não deve conter ausentes nas variáveis do modelo."
print("Base do modelo múltiplo validada.")

In [ ]:
# Ajuste o modelo múltiplo usando statsmodels.

X_multiplo = None
fit_multiplo = None

print(fit_multiplo.summary())

In [ ]:
assert X_multiplo is not None, "crie X_multiplo."
assert fit_multiplo is not None, "crie fit_multiplo."
assert hasattr(fit_multiplo, "params"), "fit_multiplo deve ser um resultado de modelo statsmodels."
for coluna in variaveis_modelo_multiplo:
    assert coluna in fit_multiplo.params.index, f"o modelo múltiplo deve conter {coluna}."
print("Modelo múltiplo validado.")

In [ ]:
# Organize os principais resultados do modelo múltiplo em uma tabela.
# A tabela deve conter uma linha por termo do modelo.
# A tabela deve conter: termo, coeficiente, erro_padrao, estatistica_t, p_t, ic_95_inf e ic_95_sup.
# Use alpha=0.05 no intervalo de confiança.
# Dicas: fit_multiplo.params, fit_multiplo.bse, fit_multiplo.tvalues,
# fit_multiplo.pvalues e fit_multiplo.conf_int(alpha=0.05).

resultados_multiplo = None

resultados_multiplo


In [ ]:
assert isinstance(resultados_multiplo, pd.DataFrame), "resultados_multiplo deve ser um DataFrame."
for coluna in colunas_resultados_modelo:
    assert coluna in resultados_multiplo.columns, f"resultados_multiplo deve conter {coluna}."
termos_multiplo = set(resultados_multiplo["termo"])
assert "const" in termos_multiplo, "inclua const em resultados_multiplo."
for coluna in variaveis_modelo_multiplo:
    assert coluna in termos_multiplo, f"inclua {coluna} em resultados_multiplo."
for coluna in colunas_resultados_modelo[1:]:
    assert pd.api.types.is_numeric_dtype(resultados_multiplo[coluna]), f"{coluna} deve ser numérica."
assert resultados_multiplo["p_t"].between(0, 1).all(), "p_t deve estar entre 0 e 1."
print("Tabela do modelo múltiplo validada.")


In [ ]:
# Compare o modelo simples e o modelo múltiplo.
# A tabela deve conter R2, R2 ajustado, erro padrão residual e os resultados inferenciais de pontos.
# Inclua, para pontos: coeficiente, erro padrão, estatística t, valor-p p(t) e IC 95%.

comparacao_modelos = None

comparacao_modelos


In [ ]:
assert isinstance(comparacao_modelos, pd.DataFrame), "comparacao_modelos deve ser um DataFrame."
colunas_comparacao = [
    "modelo",
    "R2",
    "R2_ajustado",
    "erro_padrao_residual",
    "coef_pontos",
    "erro_padrao_pontos",
    "t_pontos",
    "p_t_pontos",
    "ic_95_pontos_inf",
    "ic_95_pontos_sup",
]
for coluna in colunas_comparacao:
    assert coluna in comparacao_modelos.columns, f"comparacao_modelos deve conter {coluna}."
assert len(comparacao_modelos) == 2, "compare exatamente o modelo simples e o modelo múltiplo."
assert comparacao_modelos["p_t_pontos"].between(0, 1).all(), "p_t_pontos deve estar entre 0 e 1."
print("Comparação dos modelos validada.")


### **Interpretação Comparativa dos Modelos**

Responda em 10 a 14 linhas:

- o modelo múltiplo aumentou o $R^2$ e o $R^2$ ajustado?
- o erro padrão residual diminuiu?
- o coeficiente de `pontos` mudou após incluir as novas variáveis?
- o erro padrão de `pontos` aumentou ou diminuiu no modelo múltiplo?
- a estatística t e o valor-p `p(t)` de `pontos` mudaram de forma relevante?
- com $\alpha = 0{,}05$, a conclusão sobre associação de `pontos` com salário mudou entre os modelos?
- o IC 95% de `pontos` ficou mais estreito, mais amplo ou passou a incluir/excluir zero?
- como interpretar `pontos` no modelo simples e no modelo múltiplo?
- as variáveis escolhidas melhoraram a interpretação substantiva do salário ou apenas aumentaram o $R^2$?


## **Diagnóstico por Resíduos**

Avalie se os resíduos sugerem problemas importantes no ajuste. O objetivo é interpretar padrões básicos, não remover observações automaticamente.

In [ ]:
# Crie uma tabela com valores ajustados e resíduos dos dois modelos.

diagnostico_modelos = None

diagnostico_modelos.head()

In [ ]:
assert isinstance(diagnostico_modelos, pd.DataFrame), "diagnostico_modelos deve ser um DataFrame."
for coluna in ["modelo", "valor_ajustado", "residuo"]:
    assert coluna in diagnostico_modelos.columns, f"diagnostico_modelos deve conter {coluna}."
assert diagnostico_modelos["modelo"].nunique() == 2, "inclua os dois modelos no diagnóstico."
print("Tabela de diagnóstico validada.")

In [ ]:
# Faça gráficos de resíduos versus valores ajustados para os dois modelos.
# Use Matplotlib.
# Inclua título e rótulos dos eixos em cada painel.

# SEU CÓDIGO AQUI

In [ ]:
# Faça QQ-plots dos resíduos para os dois modelos.
# Use sm.qqplot.
# Inclua título em cada painel.

# SEU CÓDIGO AQUI

In [ ]:
# Identifique observações atípicas por resíduos padronizados internos.
# Use |resíduo padronizado| > 3 como regra operacional diagnóstica.

outliers_modelos = None

outliers_modelos

In [ ]:
assert isinstance(outliers_modelos, pd.DataFrame), "outliers_modelos deve ser um DataFrame."
for coluna in ["modelo", "n_observacoes", "n_outliers", "percentual_outliers"]:
    assert coluna in outliers_modelos.columns, f"outliers_modelos deve conter {coluna}."
print("Contagem de observações atípicas validada.")

### **Interpretação do Diagnóstico**

Responda em 6 a 10 linhas:

- os resíduos parecem aleatórios em torno de zero?
- há padrão de funil, curvatura ou pontos muito extremos?
- os QQ-plots sugerem afastamento grosseiro da normalidade?
- os outliers parecem estar associados a salários muito altos?
- o diagnóstico enfraquece alguma conclusão dos modelos?

## **Conclusão Final**

Escreva uma conclusão de 12 a 16 linhas respondendo:

- qual foi a relação observada entre `pontos` e `salario_usd`;
- se o modelo simples explicou bem a variação dos salários;
- como erro padrão, estatística t, valor-p `p(t)` e IC 95% sustentam ou enfraquecem a interpretação do coeficiente de `pontos` no modelo simples;
- quais variáveis foram escolhidas no modelo múltiplo e por quê;
- se o modelo múltiplo melhorou a interpretação;
- se a evidência inferencial de `pontos` mudou no modelo múltiplo, considerando $\alpha = 0{,}05$;
- quais limitações decorrem da unidade `jogador-temporada`;
- quais limitações decorrem de salários extremos, contratos, posição, mercado e mudanças históricas da NBA;
- por que os resultados não devem ser interpretados como causalidade.


## **Checklist Antes da Entrega**

Antes de entregar, confira:

- o notebook executa do início ao fim depois que todas as células editáveis são preenchidas;
- o grupo e o time estão corretos;
- o filtro usa todas as temporadas disponíveis do time;
- a unidade de análise foi interpretada como `jogador-temporada`;
- o modelo simples obrigatório é `salario_usd ~ pontos`;
- a tabela `resultados_simples` contém coeficiente, erro padrão, estatística t, valor-p `p(t)` e IC 95%;
- o IC 95% dos coeficientes foi interpretado separadamente do IC 95% da média ajustada;
- o modelo múltiplo contém `pontos` e pelo menos mais uma variável escolhida pelo grupo;
- a tabela `resultados_multiplo` contém coeficiente, erro padrão, estatística t, valor-p `p(t)` e IC 95%;
- há justificativa para as variáveis do modelo múltiplo;
- há contabilidade das linhas removidas por `dropna`;
- há comparação entre modelo simples e modelo múltiplo usando $R^2$, $R^2$ ajustado, erro padrão residual e métricas inferenciais de `pontos`;
- há diagnóstico por resíduos;
- todos os gráficos obrigatórios foram construídos;
- todos os gráficos têm título, rótulos nos eixos e interpretação textual logo abaixo;
- não há uso de `seaborn`;
- as interpretações não afirmam causalidade.
